# Imports

In [1]:
import os
import sys
import glob
sys.path.insert(0, "/project/def-nahee/kbas/Graphnet-Applications/Metadata")
import paths
from icecube import icetray

import pandas as pd
from collections import Counter
from icecube import dataio


from icecube import LeptonInjector, simclasses 



icetray.I3Logger.global_logger = icetray.I3NullLogger()

# Settings

In [2]:
DATASET = "SPRING2026MC"

# Available Datasets

In [3]:
all_datasets = {
    name: val for name, val in vars(paths).items()
    if not name.startswith("_") and isinstance(val, dict)
    and all(isinstance(v, dict) for v in val.values())
}

for ds_name, ds in all_datasets.items():
    for geometry, flavors in ds.items():
        for flavor, info in flavors.items():
            if isinstance(info, dict):
                status = info["path"] if info.get("path") else "None"
                print(f"[{ds_name}] {geometry}/{flavor}: {status}")

[SPRING2026MC_I3] full_geometry/Muon: /project/6008051/pone_simulation/MC000008-nu_mu-2_6-LeptonInjector_PROPOSAL_clsim-v17.1/Generator
[SPRING2026MC_I3] full_geometry/Electron: /project/6008051/pone_simulation/MC000009-nu_e-2_6-LeptonInjector_PROPOSAL_clsim-v17.1/Generator
[SPRING2026MC_I3] full_geometry/Tau: /project/6008051/pone_simulation/MC000010-nu_tau-2_6-LeptonInjector_PROPOSAL_clsim-v17.1/Generator
[SPRING2026MC_I3] full_geometry/NC: /project/6008051/pone_simulation/MC000011-nu_NC-2_6-LeptonInjector_PROPOSAL_clsim_NC-v17.1/Generator
[SPRING2026MC_I3] strings_102_40m/Muon: /scratch/kbas/Spring2026MC/Strings_102_40m/Muon_I3Photons
[SPRING2026MC_I3] strings_102_40m/Electron: /scratch/kbas/Spring2026MC/Strings_102_40m/Electron_I3Photons
[SPRING2026MC_I3] strings_102_40m/Tau: /scratch/kbas/Spring2026MC/Strings_102_40m/Tau_I3Photons
[SPRING2026MC_I3] strings_102_40m/NC: /scratch/kbas/Spring2026MC/Strings_102_40m/NC_I3Photons
[SPRING2026MC_I3] strings_102_80m/Muon: /scratch/kbas/Spri

# Dataset Selection

In [4]:
selected_datasets = {
    name: val for name, val in all_datasets.items()
    if DATASET in name
}

print(f"Datasets matching '{DATASET}':")
for name in selected_datasets:
    print(f"  {name}")

Datasets matching 'SPRING2026MC':
  SPRING2026MC_I3
  SPRING2026MC_PMT


In [5]:
selected_datasets = {
    name: val for name, val in selected_datasets.items()
    if "PMT" not in name
}

print(f"After excluding PMT:")
for name in selected_datasets:
    print(f"  {name}")

After excluding PMT:
  SPRING2026MC_I3


In [6]:
print(f"Available (non-None) entries:")
for ds_name, ds in selected_datasets.items():
    for geometry, flavors in ds.items():
        for flavor, info in flavors.items():
            if isinstance(info, dict) and info.get("path") is not None:
                print(f"  [{ds_name}] {geometry}/{flavor}: {info['path']}  (format={info['format']})")

Available (non-None) entries:
  [SPRING2026MC_I3] full_geometry/Muon: /project/6008051/pone_simulation/MC000008-nu_mu-2_6-LeptonInjector_PROPOSAL_clsim-v17.1/Generator  (format=zst)
  [SPRING2026MC_I3] full_geometry/Electron: /project/6008051/pone_simulation/MC000009-nu_e-2_6-LeptonInjector_PROPOSAL_clsim-v17.1/Generator  (format=zst)
  [SPRING2026MC_I3] full_geometry/Tau: /project/6008051/pone_simulation/MC000010-nu_tau-2_6-LeptonInjector_PROPOSAL_clsim-v17.1/Generator  (format=zst)
  [SPRING2026MC_I3] full_geometry/NC: /project/6008051/pone_simulation/MC000011-nu_NC-2_6-LeptonInjector_PROPOSAL_clsim_NC-v17.1/Generator  (format=zst)
  [SPRING2026MC_I3] strings_102_40m/Muon: /scratch/kbas/Spring2026MC/Strings_102_40m/Muon_I3Photons  (format=gz)
  [SPRING2026MC_I3] strings_102_40m/Electron: /scratch/kbas/Spring2026MC/Strings_102_40m/Electron_I3Photons  (format=gz)
  [SPRING2026MC_I3] strings_102_40m/Tau: /scratch/kbas/Spring2026MC/Strings_102_40m/Tau_I3Photons  (format=gz)
  [SPRING2026

In [7]:
print("File counts:")
for ds_name, ds in selected_datasets.items():
    for geometry, flavors in ds.items():
        for flavor, info in flavors.items():
            if isinstance(info, dict) and info.get("path") is not None:
                count = len(glob.glob(os.path.join(info["path"], "**", f"*.i3.{info['format']}"), recursive=True))
                print(f"  [{ds_name}] {geometry}/{flavor}: {count} files")

File counts:


  [SPRING2026MC_I3] full_geometry/Muon: 88 files
  [SPRING2026MC_I3] full_geometry/Electron: 21 files
  [SPRING2026MC_I3] full_geometry/Tau: 28 files
  [SPRING2026MC_I3] full_geometry/NC: 46 files
  [SPRING2026MC_I3] strings_102_40m/Muon: 88 files
  [SPRING2026MC_I3] strings_102_40m/Electron: 21 files
  [SPRING2026MC_I3] strings_102_40m/Tau: 28 files
  [SPRING2026MC_I3] strings_102_40m/NC: 46 files
  [SPRING2026MC_I3] strings_102_80m/Muon: 88 files
  [SPRING2026MC_I3] strings_102_80m/Electron: 21 files
  [SPRING2026MC_I3] strings_102_80m/Tau: 28 files
  [SPRING2026MC_I3] strings_102_80m/NC: 46 files


In [8]:
representative_files = {}

for ds_name, ds in selected_datasets.items():
    for geometry, flavors in ds.items():
        for flavor, info in flavors.items():
            if isinstance(info, dict) and info.get("path") is not None:
                files = sorted(glob.glob(os.path.join(info["path"], "**", f"*.i3.{info['format']}"), recursive=True))
                key = f"{ds_name}/{geometry}/{flavor}"
                representative_files[key] = files[0] if files else None
                print(f"  {key}: {representative_files[key]}")

  SPRING2026MC_I3/full_geometry/Muon: /project/6008051/pone_simulation/MC000008-nu_mu-2_6-LeptonInjector_PROPOSAL_clsim-v17.1/Generator/33939527/gen_001.i3.zst
  SPRING2026MC_I3/full_geometry/Electron: /project/6008051/pone_simulation/MC000009-nu_e-2_6-LeptonInjector_PROPOSAL_clsim-v17.1/Generator/33996141/gen_008.i3.zst
  SPRING2026MC_I3/full_geometry/Tau: /project/6008051/pone_simulation/MC000010-nu_tau-2_6-LeptonInjector_PROPOSAL_clsim-v17.1/Generator/33996158/gen_001.i3.zst
  SPRING2026MC_I3/full_geometry/NC: /project/6008051/pone_simulation/MC000011-nu_NC-2_6-LeptonInjector_PROPOSAL_clsim_NC-v17.1/Generator/33996259/gen_001.i3.zst
  SPRING2026MC_I3/strings_102_40m/Muon: /scratch/kbas/Spring2026MC/Strings_102_40m/Muon_I3Photons/muon_33939527_gen_001.i3.gz
  SPRING2026MC_I3/strings_102_40m/Electron: /scratch/kbas/Spring2026MC/Strings_102_40m/Electron_I3Photons/electron_33996141_gen_008.i3.gz
  SPRING2026MC_I3/strings_102_40m/Tau: /scratch/kbas/Spring2026MC/Strings_102_40m/Tau_I3Phot

# File Analysis

## Frame Types

In [9]:
rows = []
for label, path in representative_files.items():
    if path is None:
        continue
    counts = Counter()
    f = dataio.I3File(path)
    while f.more():
        frame = f.pop_frame()
        counts[str(frame.Stop)] += 1
    f.close()
    row = {"file": label}
    row.update(counts)
    rows.append(row)

df_frames = pd.DataFrame(rows).fillna(0).set_index("file")
df_frames = df_frames.astype(int)
df_frames

,TrayInfo,Simulation,DAQ
file,,,
SPRING2026MC_I3/full_geometry/Muon,1,2,99
SPRING2026MC_I3/full_geometry/Electron,1,2,200
SPRING2026MC_I3/full_geometry/Tau,1,2,200
SPRING2026MC_I3/full_geometry/NC,1,2,199
SPRING2026MC_I3/strings_102_40m/Muon,0,0,29
SPRING2026MC_I3/strings_102_40m/Electron,0,0,58
SPRING2026MC_I3/strings_102_40m/Tau,0,0,55
SPRING2026MC_I3/strings_102_40m/NC,0,0,57
SPRING2026MC_I3/strings_102_80m/Muon,0,0,58


## TrayInfo Frames

In [10]:
for label, path in representative_files.items():
    if path is None:
        continue
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")
    f = dataio.I3File(path)
    while f.more():
        frame = f.pop_frame()
        if frame.Stop == icetray.I3Frame.TrayInfo:
            for key in frame.keys():
                print(frame[key])
            break
    f.close()


SPRING2026MC_I3/full_geometry/Muon
============================= configuration ==============================
      Icetray run started:  Fri Apr 10 11:45:29 2026
                  on host:  fc11004 (Linux-5.14.0-611.30.1.el9_7.x86_64/x86_64/gcc-13.3.0)
                  as user:  cmiller
          and gcc version:  13.3.0
                  svn url:  
             svn revision:  0

===========================   svn:externals   ============================

=============================   services    ==============================
Earth (I3EarthModelServiceFactory)
  DetectorDepth
    Description :  Depth of origin of IceCube coordinate, measured from ice surface
    Default     :  1948.0
    Configured  :  2100.0

  EarthModels
    Description :  vector of basenames of EarthModel param file. ex. PREM_mmc.dat -> PREM_mmc suffix of the param file must be .dat 
    Default     :  []
    Configured  :  ['PREM_pone']

  IceCapSimpleAngle
    Description :  Valid only when IceCapType is 'Si

## Simulation Frames

In [11]:
for label, path in representative_files.items():
    if path is None:
        continue
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")
    f = dataio.I3File(path)
    sim_count = 0
    while f.more():
        frame = f.pop_frame()
        if str(frame.Stop) == "Simulation":
            sim_count += 1
            print(f"\n--- Simulation frame {sim_count} ---")
            for key in frame.keys():
                print(f"  {key}: {type(frame[key]).__name__}")
                print(f"    {frame[key]}")
    f.close()


SPRING2026MC_I3/full_geometry/Muon

--- Simulation frame 1 ---
  LeptonInjectorProperties: RangedInjectionConfiguration


IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)




SPRING2026MC_I3/full_geometry/Electron

--- Simulation frame 1 ---
  LeptonInjectorProperties: VolumeInjectionConfiguration


IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)




SPRING2026MC_I3/full_geometry/Tau

--- Simulation frame 1 ---
  LeptonInjectorProperties: VolumeInjectionConfiguration


IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)




SPRING2026MC_I3/full_geometry/NC

--- Simulation frame 1 ---
  LeptonInjectorProperties: VolumeInjectionConfiguration


IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)




SPRING2026MC_I3/strings_102_40m/Muon

SPRING2026MC_I3/strings_102_40m/Electron

SPRING2026MC_I3/strings_102_40m/Tau

SPRING2026MC_I3/strings_102_40m/NC

SPRING2026MC_I3/strings_102_80m/Muon

SPRING2026MC_I3/strings_102_80m/Electron

SPRING2026MC_I3/strings_102_80m/Tau

SPRING2026MC_I3/strings_102_80m/NC


## DAQ Frame Keys

In [12]:
daq_keys = {}

for label, path in representative_files.items():
    if path is None:
        continue
    f = dataio.I3File(path)
    while f.more():
        frame = f.pop_frame()
        if frame.Stop == icetray.I3Frame.DAQ:
            daq_keys[label] = set(frame.keys())
            break
    f.close()

all_keys = sorted(set(k for keys in daq_keys.values() for k in keys))
rows = []
for label, keys in daq_keys.items():
    row = {"file": label}
    for k in all_keys:
        row[k] = "✓" if k in keys else "✗"
    rows.append(row)

pd.DataFrame(rows).set_index("file")

,Accepted_PulseMap,EventProperties,I3EventHeader,I3MCTree,I3MCTree_RNGState,I3Photons,LeptonInjectorProperties,MMCTrackList,Noise_Dark,Noise_K40
file,,,,,,,,,,
SPRING2026MC_I3/full_geometry/Muon,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
SPRING2026MC_I3/full_geometry/Electron,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
SPRING2026MC_I3/full_geometry/Tau,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
SPRING2026MC_I3/full_geometry/NC,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
SPRING2026MC_I3/strings_102_40m/Muon,✓,✓,✓,✓,✓,✓,✗,✓,✓,✓
SPRING2026MC_I3/strings_102_40m/Electron,✓,✓,✓,✓,✓,✓,✗,✓,✓,✓
SPRING2026MC_I3/strings_102_40m/Tau,✓,✓,✓,✓,✓,✓,✗,✓,✓,✓
SPRING2026MC_I3/strings_102_40m/NC,✓,✓,✓,✓,✓,✓,✗,✓,✓,✓
SPRING2026MC_I3/strings_102_80m/Muon,✓,✓,✓,✓,✓,✓,✗,✓,✓,✓
